# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Answer


1. Physical Likelihood Formulation

At time step $k$, a physical sensor records a continuous measurement $Y_k = y_k$ (such as dynamic modal frequency, strain measurement, or vibrational response).

The measurement model is governed by a known structural forward response function $g(\theta)$ subject to additive Gaussian sensor noise $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$:

$$Y_k = g(\theta) + \epsilon_k$$

Conditional on the underlying structural integrity parameter $\Theta = \theta$, the likelihood contribution of a single observation $y_k$ at step $k$ is given by the Gaussian probability density function:

$$L(y_k \mid \theta) = f_{Y_k \mid \Theta}(y_k \mid \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\left(y_k - g(\theta)\right)^2}{2\sigma^2} \right)$$

---

2. Sequential Likelihood and Joint History

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)^T$ represent the vector of sequential sensor observations gathered up to step $k$.

Assuming that sensor noise terms across consecutive time steps are conditionally independent given $\Theta = \theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \frac{1}{(2\pi\sigma^2)^{k/2}} \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^k \left(y_i - g(\theta)\right)^2 \right)$$

---

3. Mathematical Formulation of Bounded Recursive Updates

Before observing measurements, the platform initializes a prior density function $f_{\Theta}^{(0)}(\theta)$ over the physically bounded interval $[\theta_{\min}, \theta_{\max}]$ (e.g., a uniform distribution $U(\theta_{\min}, \theta_{\max})$ indicating initial uninformative uncertainty).

Under a sequential Bayesian updating framework, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$. The running posterior density function $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ is recursively updated via Bayes' Theorem:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \, f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

*Definition of Key Components*:

* **$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$**: The prior density at step $k$ inherited directly from the previous state $k-1$.
* **$L(y_k \mid \theta)$**: The likelihood contribution of the incoming real-time sensor reading $y_k$.
* **Denominator (Normalizing Constant $Z_k$)**: Integrates the product of likelihood and prior across the bounded domain $[\theta_{\min}, \theta_{\max}]$ to ensure the total area under the density curve equals $1$.

---

4. Running Point Estimators

From the running posterior distribution $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, two primary estimators track structural health at step $k$:

a. Running Posterior Mean ($\widehat{\theta}_{\text{Bayes}}^{(k)}$)
Under a squared-error loss function, the optimal point estimate is the expected value of the current bounded posterior distribution:

$$\widehat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}\right] = \int_{\theta_{\min}}^{\theta_{\max}} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

b.. Running Maximum A Posteriori ($\widehat{\theta}_{\text{MAP}}^{(k)}$)
The most probable structural state corresponds to the peak (mode) of the current posterior density over the bounded domain:

$$\widehat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in [\theta_{\min}, \theta_{\max}]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

5. Numerical Implementation via Bounded Grid Discretization

Since non-linear structural response functions $g(\theta)$ generally lack closed-form analytical conjugate solutions, the system maintains the posterior on a fine discrete grid across the bounded physical domain $[\theta_{\min}, \theta_{\max}]$.

*Algorithmic Procedure*:

a. **Grid Setup:** Define $M$ equally spaced grid points over $[\theta_{\min}, \theta_{\max}]$:
   $$\theta_m = \theta_{\min} + (m-1)\Delta\theta, \quad \text{where } \Delta\theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}, \quad m = 1, 2, \dots, M$$

b. **Prior Initialization:** Evaluate initial prior values across the grid array $\mathbf{P}_0 = [P_0(\theta_1), \dots, P_0(\theta_M)]$ and normalize using numerical integration (e.g., composite trapezoidal rule):
   $$Z_0 = \text{trapezoid}(\mathbf{P}_0, \boldsymbol{\theta}), \quad \mathbf{P}_0 \leftarrow \frac{\mathbf{P}_0}{Z_0}$$

c. **Sequential Updating & Normalization (at step $k$):**
   * Compute unnormalized posterior array: $\tilde{P}_k(\theta_m) = P_{k-1}(\theta_m) \times L(y_k \mid \theta_m)$
   * Compute normalizing factor: $Z_k = \sum_{m=1}^{M-1} \frac{\tilde{P}_k(\theta_m) + \tilde{P}_k(\theta_{m+1})}{2} \Delta\theta$
   * Normalize: $P_k(\theta_m) = \frac{\tilde{P}_k(\theta_m)}{Z_k}$

d. **Point Estimation Evaluation:**
   * **Bayes Estimate:** $\widehat{\theta}_{\text{Bayes}}^{(k)} \approx \text{trapezoid}(\boldsymbol{\theta} \odot \mathbf{P}_k, \boldsymbol{\theta})$
   * **MAP Estimate:** $\widehat{\theta}_{\text{MAP}}^{(k)} = \theta_{m^*}, \quad \text{where } m^* = \arg\max_{m} P_k(\theta_m)$

---

6. Dynamic Mechanics & Convergence Interpretation

* **Variance Reduction:** As $k$ increases, repeated noisy sensor measurements progressively attenuate likelihood ambiguity, tightening the posterior distribution variance around the true structural state $\theta_{\text{true}}$.
* **Physical Boundary Enforcement:** The bounded grid framework strictly guarantees that probability mass outside $[\theta_{\min}, \theta_{\max}]$ is zero, avoiding physically unfeasible state estimates (such as negative stiffness or over-100% health).
* **Sensor Noise vs. Tracking Sensitivity:** Lower measurement noise $\sigma$ narrows the single-step likelihood curve $L(y_k \mid \theta)$, allowing rapid posterior convergence with fewer sensor readings.